# 3. Gating: label windows bound / confined / exploratory / transport

Two gates on the window features (`configs/gates.yaml`):

- **Gate 1**: `avg_speed_um_s` vs `self_intersections` (scaled 0–1) → bound, exploratory, or the slow/low-intersection parent.
- **Gate 2**: `anomalous_exponent` vs `cum_displacement_um`, only for the parent → confined or transport.

`final_population` uses these gates. `final_population_certain` uses tighter "certain" gates: a window keeps
its label only if both agree, otherwise it is `unknown`. The transformer is trained on `final_population_certain`.

The gate geometry and the gate-1 scaler are fixed in `gates.yaml`, so new data is labelled exactly like the paper data.

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import sptnano` works from this folder

import polars as pl

from sptnano import config

paths = config.load_paths()
settings = config.load_settings()

from datetime import date

from sptnano import gating, master_tables

## 1. Load the master tables and remove bad fits

In [ ]:
instant_df, windowed_df = master_tables.load_master_tables(paths["master_tables_dir"])

# Raw-movie paths were saved on Z:, the data now lives on F:.
for drive_fix in [("Z:", "F:")]:
    instant_df = instant_df.with_columns(pl.col("foldername").str.replace(*drive_fix, literal=True))
    windowed_df = windowed_df.with_columns(pl.col("foldername").str.replace(*drive_fix, literal=True))

print("windowed before removing bad fits:", windowed_df.shape)
windowed_df = windowed_df.filter(pl.col("bad_fit_flag") == False)  # noqa: E712
instant_df = instant_df.filter(pl.col("window_uid").is_in(windowed_df["window_uid"].unique()))
print("windowed:", windowed_df.shape, " instant:", instant_df.shape)

## 2. Gate

In [ ]:
gated_windowed = gating.gate_windows(windowed_df)
gated_instant = gating.label_instant_rows(instant_df, gated_windowed)

display(gating.label_counts(gated_windowed, "final_population"))
display(gating.label_counts(gated_windowed, "final_population_certain"))

## 3. Gate plots

Hexbin of every window, coloured by molecule, with the gates drawn on top.

In [ ]:
plot_dir = paths["plots_dir"] / "gating"
plot_dir.mkdir(parents=True, exist_ok=True)

gates = config.load_gates()
gating.plot_gates(gated_windowed, "gate1", save_dir=plot_dir)
gating.plot_gates(gated_windowed, "certain_gate1", save_dir=plot_dir)

# Gate 2 only shows the windows inside its parent gate-1 region.
gating.plot_gates(gated_windowed.filter(pl.col("gate_name") == gates["gate2"]["parent"]), "gate2", save_dir=plot_dir)
gating.plot_gates(
    gated_windowed.filter(pl.col("certain_gate_name") == gates["certain_gate2"]["parent"]), "certain_gate2", save_dir=plot_dir
)

## 4. Save

Writes to a new dated folder so the gated tables in use are never overwritten.
Point `gated_dir` in `configs/paths.local.yaml` at it when you want to use it.

In [ ]:
out_dir = paths["gated_dir"].parent / f"HIERARCHICAL_GATES_CERTAIN_{date.today():%Y%m%d}"
gating.save_gated_tables(gated_windowed, gated_instant, out_dir)